# 09 — LoRA fine-tune of Phi-3.5-mini for D.R.O.N.A. advising (Colab T4)

**Phase 3.** QLoRA (4-bit) + PEFT fine-tune of `microsoft/Phi-3.5-mini-instruct` on the
synthetic, Nepal-grounded advising Q&A dataset.

This notebook is **Colab-T4 shaped** (it will NOT run on the student's GTX 1650 4 GB).
Runtime → Change runtime type → **T4 GPU**.

Pipeline: install → build SFT data with `drona.finetune` → load Phi-3.5 in 4-bit → train
rank-16 LoRA → save/merge → base-vs-LoRA sanity check.

Every hyperparameter lives once in `drona.finetune.lora_config.DronaLoraConfig` so the
notebook and repo never drift. Refs: QLoRA (Dettmers et al. 2023).

## 1. Install dependencies + clone the repo

In [ ]:
!pip -q install transformers>=4.44 peft>=0.12 trl>=0.9.6 accelerate>=0.33 datasets>=2.18 bitsandbytes>=0.43
# Clone DRONA so we can reuse the data-prep + config code (replace with your URL/path).
import os
if not os.path.exists('D.R.O.N.A'):
    !git clone https://github.com/your-username/D.R.O.N.A.git || echo 'clone skipped; using uploaded files'
%cd /content/D.R.O.N.A 2>/dev/null || echo 'run from repo root'
!pip -q install -e . || pip -q install pydantic loguru

## 2. Build (or upload) the SFT dataset

Either generate it here from real career-pathway anchors, or upload
`data/finetune/sft_train.jsonl` / `sft_val.jsonl` produced locally by
`python scripts/generate_qa.py --pathways data/processed/onet_career_pathways.json --n 500`.

In [ ]:
import json, pathlib
from drona.finetune import qa_generator, dataset as ds
from drona.contracts import CareerPathway

anchors_path = pathlib.Path('data/processed/onet_career_pathways.json')
if anchors_path.exists():
    pathways = [CareerPathway.model_validate(r) for r in json.loads(anchors_path.read_text())]
    pairs = qa_generator.generate_qa_pairs(pathways, target_count=500)
    examples = ds.build_sft_dataset(pairs)
    train, val = ds.train_val_split(examples)
    ds.export_jsonl(train, pathlib.Path('data/finetune/sft_train.jsonl'))
    ds.export_jsonl(val, pathlib.Path('data/finetune/sft_val.jsonl'))
else:
    train = ds.load_jsonl(pathlib.Path('data/finetune/sft_train.jsonl'))
    val = ds.load_jsonl(pathlib.Path('data/finetune/sft_val.jsonl'))
print(f'train={len(train)} val={len(val)}')
print(train[0]['text'][:400])

In [ ]:
from datasets import Dataset
# trl SFTTrainer (chat mode) consumes the 'messages' field directly.
train_ds = Dataset.from_list([{'messages': e['messages']} for e in train])
val_ds = Dataset.from_list([{'messages': e['messages']} for e in val])
train_ds

## 3. Load Phi-3.5-mini in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from drona.finetune.lora_config import DronaLoraConfig

cfg = DronaLoraConfig()
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    quantization_config=cfg.to_bnb_config(),
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
print(model.get_memory_footprint() / 1e9, 'GB')

## 4. Train the LoRA adapter

In [ ]:
from peft import prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

model = prepare_model_for_kbit_training(model)
peft_cfg = cfg.to_peft_config()

sft_args = SFTConfig(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    warmup_ratio=cfg.warmup_ratio,
    logging_steps=cfg.logging_steps,
    save_strategy=cfg.save_strategy,
    max_seq_length=cfg.max_seq_length,
    bf16=True,
    seed=cfg.seed,
    report_to=[],
)
trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_cfg,
    processing_class=tokenizer,
)
trainer.train()

## 5. Save adapter + serve via Ollama

Save the LoRA adapter, optionally merge into the base weights, then convert the
merged model to GGUF (`llama.cpp convert-hf-to-gguf.py`) and `ollama create` a
model tag. The advising request path stays **local** (C4).

In [ ]:
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print('Adapter saved to', cfg.output_dir)
# Merge for GGUF export (optional, needs full-precision reload):
# from peft import AutoPeftModelForCausalLM
# merged = AutoPeftModelForCausalLM.from_pretrained(cfg.output_dir, torch_dtype='auto')
# merged = merged.merge_and_unload(); merged.save_pretrained(cfg.output_dir + '-merged')
# Then: python llama.cpp/convert_hf_to_gguf.py <merged> --outfile phi35-advising.gguf --outtype q4_k_m
# And:  ollama create phi35-advising -f Modelfile

## 6. Base vs LoRA sanity check

The full transparent ablation (pathway count, grounded rate, Nepal-citation ratio,
hedge frequency, refusal rate) lives in `drona.finetune.ablation.run_ablation`
and runs both models WITH the RAG pipeline. Below is a single-prompt smoke check.

In [ ]:
from transformers import pipeline
gen = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=400)
msg = train[0]['messages'][:2]
out = gen(tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True))
print(out[0]['generated_text'][-800:])